# データを表として見る

価格と需給計画の基本表を読むための入口です。ローカルの保存済みデータだけを読みます。
ファイルへの書込み、再取得、補間、headroom計算、図・回帰は行いません。
案Aのsnapshotや端点のbaselineも選択しません。

VS Codeではrepoの `.venv` をカーネルとして選び、上から実行してください。


In [ ]:
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq

# このノートブックのカーネルの作業場所からrepoを探す。
roots = [Path.cwd(), *Path.cwd().parents]
repo = next((p for p in roots if (p / "AGENTS.md").exists()), None)
if repo is None:
    raise FileNotFoundError("このrepoのフォルダ内から実行してください。")
data_dir = repo / "data/processed/occto/demand_supply_direction/20261016_native_v1"
if not data_dir.is_dir():
    raise FileNotFoundError(
        "ローカルの実データがありません。共有repoのdata/を確認してください。"
        "GitHubからのcloneだけでは実データは含まれません。"
    )
print(f"参照フォルダ: {data_dir}")


## 何が保存されているか

基本市場表、案A4条件の市場表、需要3表、計画需給2表、連系線3表です。
行数は保存済みファイルのメタデータから読みます。


In [ ]:
inventory = []
for path in sorted(data_dir.glob("*.parquet")):
    stored = pq.ParquetFile(path)
    inventory.append({"ファイル": path.name, "行数": stored.metadata.num_rows,
                      "列数": len(stored.schema_arrow)})
display(pd.DataFrame(inventory))


## 価格と需給計画を先頭から見る

`area_price` は地域固有の実績価格、system priceと価格感応度は全国共通系列を地域ごとに反復した値です。
`reserve_nextday2__` は翌々日の計画断面、`reserve_latest__` は翌日・当日の計画snapshotです。
両者とも実績需要の代わりにはしません。価格単位は円/kWh、下の需要・供給力列はMWです。


In [ ]:
market_file = data_dir / "market_base_FY2025_with_reserve.parquet"
columns = [
    "area", "delivery_date", "period", "timestamp_start", "timestamp_end",
    "area_price", "system_price_market", "system_price_sensitivity_base",
    "baseline_price_match_flag",
    "reserve_nextday2__area_demand_forecast_mw",
    "reserve_latest__area_demand_forecast_mw",
    "reserve_latest__area_supply_plan_mw",
]
stored = pq.ParquetFile(market_file)
print(f"基本市場表: {stored.metadata.num_rows:,}行、{len(stored.schema_arrow)}列")
# 表示用に最初の20行だけを読み、原ファイルには触れない。
preview = next(stored.iter_batches(batch_size=20, columns=columns)).to_pandas()
display(preview)


## 詳細へ進む

- `../docs/data_guide.md`：各ファイルと変数の読み方。
- `../reports/README.md`：工程別の報告書。
- `../metadata/variable_dictionary.csv`：全変数の単位と定義。

参照容量差や公表記録を見る場合は、案A4ファイルのうち比較対象を明示して開いてください。
4条件を独立した市場観測として足しません。ここでは1条件を自動選択していません。
